# DDIM — Denoising Diffusion Implicit Models
### From Scratch Implementation

---

## What is DDIM?

DDIM (Song et al. 2020) is a **faster sampling method** for diffusion models.
It uses the **same trained model as DDPM** but changes only the sampling process.

---

## How DDIM differs from DDPM

| | DDPM | DDIM |
|---|---|---|
| Sampling steps | 1000 | 50 (or less) |
| Random noise added back | ✅ Yes | ❌ No |
| Deterministic | ❌ No | ✅ Yes |
| Needs retraining | — | ❌ Same model |
| Speed | slow | ~20x faster |
| Quality | good | same or better |

---

## Key Idea

**DDPM** at each step:
```
1. predict noise ε
2. remove noise → get mean
3. add random noise back  ← this forces 1000 steps
```

**DDIM** at each step:
```
1. predict noise ε
2. estimate clean image x0 right now  ← NEW
3. jump to any timestep directly       ← NEW
4. NO random noise added back          ← KEY DIFFERENCE
```

---

## DDIM Core Formula

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \underbrace{\frac{x_t - \sqrt{1-\bar\alpha_t} \cdot \epsilon_\theta}{\sqrt{\bar\alpha_t}}}_{\text{predicted } x_0} + \underbrace{\sqrt{1-\bar\alpha_{t-1}} \cdot \epsilon_\theta}_{\text{direction to } x_t}$$

**Step 1 — Estimate x0:**
$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t} \cdot \epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}$$

**Step 2 — Jump to previous timestep:**
$$x_{t_{prev}} = \sqrt{\bar\alpha_{t_{prev}}} \cdot \hat{x}_0 + \sqrt{1-\bar\alpha_{t_{prev}}} \cdot \epsilon_\theta$$

---

## Why Skipping is Possible in DDIM

Because DDIM is **deterministic** (no random noise), given the same `xt` it always produces the same `x_{t-1}`. This means we can compute where any `x_t` will end up **without visiting every step in between**.

DDPM adds random noise → **unpredictable** → must visit every step  
DDIM adds no random noise → **predictable** → can skip steps safely

---
## Step 1 — Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Step 2 — Noise Schedule

Same as DDPM — DDIM reuses everything from DDPM training.

In [ ]:
# ── Noise schedule (same as DDPM) ────────────────────────
T = 1000  # total timesteps

betas = torch.linspace(1e-4, 0.02, T).to(device)          # β_1 ... β_T
alphas = 1.0 - betas                                        # α_t = 1 - β_t
alphas_cumprod = torch.cumprod(alphas, dim=0)               # ᾱ_t
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)  # ᾱ_{t-1}

# Precompute values used in forward + reverse process
sqrt_alphas_cumprod       = torch.sqrt(alphas_cumprod)       # √ᾱ_t
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)  # √(1-ᾱ_t)
sqrt_recip_alphas         = torch.sqrt(1.0 / alphas)         # 1/√α_t
sqrt_recip_alphas_cumprod = torch.sqrt(1.0 / alphas_cumprod) # 1/√ᾱ_t  ← needed for DDIM

print(f"Beta range: {betas[0]:.5f} → {betas[-1]:.5f}")
print(f"ᾱ_t range:  {alphas_cumprod[0]:.5f} → {alphas_cumprod[-1]:.5f}")

---
## Step 3 — Helper Functions

In [ ]:
def extract(a, t, x_shape):
    """Extract values from schedule tensor at timestep t and reshape for broadcasting."""
    B = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(B, *((1,) * (len(x_shape) - 1)))


def q_sample(x0, t, noise=None):
    """Forward process: add noise to x0 at timestep t in ONE step.
    
    Formula: xt = √ᾱ_t * x0 + √(1-ᾱ_t) * ε
    """
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_ab_t    = extract(sqrt_alphas_cumprod,           t, x0.shape)  # √ᾱ_t
    sqrt_1mab_t  = extract(sqrt_one_minus_alphas_cumprod, t, x0.shape)  # √(1-ᾱ_t)

    return sqrt_ab_t * x0 + sqrt_1mab_t * noise, noise

---
## Step 4 — U-Net Model (same as DDPM)

DDIM uses the **exact same U-Net** — no changes to architecture.

In [ ]:
# ── Sinusoidal Time Embedding ────────────────────────────
class SinusoidalPE(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half   = self.dim // 2
        freqs  = torch.exp(
            -torch.arange(half, device=device) * (np.log(10000) / (half - 1))
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


# ── Residual Block ───────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1  = nn.GroupNorm(8, in_ch)
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.norm2  = nn.GroupNorm(8, out_ch)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(F.silu(t_emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


# ── Simple U-Net ─────────────────────────────────────────
class SimpleUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, t_dim=128):
        super().__init__()
        self.t_mlp = nn.Sequential(
            SinusoidalPE(t_dim),
            nn.Linear(t_dim, t_dim * 4),
            nn.SiLU(),
            nn.Linear(t_dim * 4, t_dim),
        )
        ch = base_ch

        # Encoder
        self.init_conv = nn.Conv2d(img_channels, ch, 3, padding=1)
        self.pool      = nn.MaxPool2d(2)
        self.down1     = ResBlock(ch,   ch*2, t_dim)
        self.down2     = ResBlock(ch*2, ch*4, t_dim)

        # Bottleneck
        self.bot1 = ResBlock(ch*4, ch*4, t_dim)
        self.bot2 = ResBlock(ch*4, ch*4, t_dim)

        # Decoder — skip connections paired by matching spatial size
        self.up1 = ResBlock(ch*4 + ch*2, ch*2, t_dim)  # cat(up(b), x2)
        self.up2 = ResBlock(ch*2 + ch,   ch,   t_dim)  # cat(up(h), x1)
        self.up  = nn.Upsample(scale_factor=2, mode='nearest')

        self.out = nn.Sequential(
            nn.GroupNorm(8, ch),
            nn.SiLU(),
            nn.Conv2d(ch, img_channels, 1),
        )

    def forward(self, x, t):
        t_emb = self.t_mlp(t)

        # Encoder
        x1 = self.init_conv(x)                    # [B, ch,   32, 32]
        x2 = self.down1(self.pool(x1), t_emb)     # [B, ch*2, 16, 16]
        x3 = self.down2(self.pool(x2), t_emb)     # [B, ch*4,  8,  8]

        # Bottleneck
        b  = self.bot2(self.bot1(x3, t_emb), t_emb)

        # Decoder
        h  = self.up1(torch.cat([self.up(b), x2], dim=1), t_emb)  # 8→16, cat x2
        h  = self.up2(torch.cat([self.up(h), x1], dim=1), t_emb)  # 16→32, cat x1
        return self.out(h)


# Initialize model
model = SimpleUNet(img_channels=1, base_ch=64).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Sanity check
x_test = torch.randn(2, 1, 32, 32).to(device)
t_test = torch.randint(0, T, (2,)).to(device)
assert model(x_test, t_test).shape == x_test.shape
print("Model forward pass OK ✅")

---
## Step 5 — Training (same as DDPM)

Training is **100% identical** to DDPM. Same loss, same optimizer, same everything.

$$L = ||\epsilon - \epsilon_\theta(x_t, t)||^2$$

In [ ]:
# ── Dataset ──────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # scale to [-1, 1]
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader  = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=4)
print(f"Dataset size: {len(dataset):,} images")

In [ ]:
# ── Training loop (identical to DDPM) ────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
EPOCHS    = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x0, _ in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x0 = x0.to(device)
        B  = x0.shape[0]

        # 1. Sample random timesteps
        t = torch.randint(0, T, (B,), device=device).long()

        # 2. Add noise — forward process
        noise = torch.randn_like(x0)
        xt, _ = q_sample(x0, t, noise)

        # 3. Predict noise
        pred_noise = model(xt, t)

        # 4. MSE loss — same as DDPM
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg:.4f}")

    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f"ddim_model_epoch{epoch+1}.pt")
        print(f"  Checkpoint saved.")

---
## Step 6 — DDIM Sampling

This is where DDIM differs from DDPM. Two things change:

**1. Timestep list** — pick evenly spaced subset instead of all 1000:
```
DDPM: [999, 998, 997, ..., 1, 0]   → 1000 steps
DDIM: [999, 979, 959, ..., 19, 0]  → 50 steps
```

**2. Each step** — estimate x0 first, then jump (no random noise):
```
DDPM: mean + random_noise           → stochastic
DDIM: √ᾱ_{t-1} * pred_x0 + √(1-ᾱ_{t-1}) * ε_θ  → deterministic
```

In [ ]:
# ── DDIM single step ─────────────────────────────────────
@torch.no_grad()
def ddim_step(model, xt, t, t_prev):
    """
    One DDIM denoising step: predict x_{t_prev} from x_t
    
    Args:
        xt:     noisy image at timestep t
        t:      current timestep tensor
        t_prev: previous (less noisy) timestep tensor, or None at final step
    """
    # Extract schedule values at current timestep t
    sqrt_ab_t    = extract(sqrt_alphas_cumprod,           t, xt.shape)  # √ᾱ_t
    sqrt_1mab_t  = extract(sqrt_one_minus_alphas_cumprod, t, xt.shape)  # √(1-ᾱ_t)

    # Step 1 — U-Net predicts noise (same as DDPM)
    pred_noise = model(xt, t)                                            # ε_θ(xt, t)

    # Step 2 — Estimate clean image x0 from current xt
    # Rearranging forward formula: xt = √ᾱ_t*x0 + √(1-ᾱ_t)*ε
    # → x0 = (xt - √(1-ᾱ_t)*ε) / √ᾱ_t
    pred_x0 = (xt - sqrt_1mab_t * pred_noise) / sqrt_ab_t
    pred_x0 = pred_x0.clamp(-1, 1)                                      # keep in valid range

    # Final step — just return predicted x0
    if t_prev is None:
        return pred_x0

    # Step 3 — Jump to t_prev using estimated x0
    # x_{t_prev} = √ᾱ_{t_prev} * pred_x0  +  √(1-ᾱ_{t_prev}) * ε_θ
    sqrt_ab_prev   = extract(sqrt_alphas_cumprod,           t_prev, xt.shape)  # √ᾱ_{t-1}
    sqrt_1mab_prev = extract(sqrt_one_minus_alphas_cumprod, t_prev, xt.shape)  # √(1-ᾱ_{t-1})

    x_prev = sqrt_ab_prev * pred_x0 + sqrt_1mab_prev * pred_noise       # NO random noise!
    return x_prev


# ── DDIM full sampling loop ───────────────────────────────
@torch.no_grad()
def sample_ddim(model, img_size=32, channels=1, n=16, ddim_steps=50):
    """
    Generate images using DDIM sampling.
    
    Args:
        ddim_steps: number of steps (50 instead of 1000)
    """
    model.eval()

    # Start from pure Gaussian noise
    x = torch.randn(n, channels, img_size, img_size).to(device)

    # Build evenly spaced timestep subsequence
    # e.g. ddim_steps=50 → [999, 979, 959, ..., 19, 0]
    timesteps = torch.linspace(0, T - 1, ddim_steps).long().flip(0).to(device)
    print(f"DDIM timesteps (first 5): {timesteps[:5].tolist()}")
    print(f"DDIM timesteps (last  5): {timesteps[-5:].tolist()}")

    for i, t_now in enumerate(tqdm(timesteps, desc=f"DDIM Sampling ({ddim_steps} steps)")):
        # Current timestep for all images in batch
        t_tensor = torch.full((n,), t_now, device=device, dtype=torch.long)

        # Previous (less noisy) timestep
        if i + 1 < len(timesteps):
            t_prev = timesteps[i + 1]
            t_prev_tensor = torch.full((n,), t_prev, device=device, dtype=torch.long)
        else:
            t_prev_tensor = None  # final step

        x = ddim_step(model, x, t_tensor, t_prev_tensor)

    # Rescale from [-1, 1] → [0, 1]
    x = (x.clamp(-1, 1) + 1) / 2
    return x

---
## Step 7 — DDPM Sampling (for comparison)

Original DDPM sampler — runs 1000 steps with random noise.

In [ ]:
# ── DDPM single step ─────────────────────────────────────
@torch.no_grad()
def p_sample_ddpm(model, x, t, t_idx):
    """One DDPM denoising step."""
    betas_t        = extract(betas,                       t, x.shape)
    sqrt_1mab_t    = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_a_t = extract(sqrt_recip_alphas,           t, x.shape)

    pred_noise = model(x, t)
    mean = sqrt_recip_a_t * (x - betas_t / sqrt_1mab_t * pred_noise)

    if t_idx == 0:
        return mean
    return mean + torch.sqrt(betas_t) * torch.randn_like(x)  # add random noise


# ── DDPM full sampling loop ───────────────────────────────
@torch.no_grad()
def sample_ddpm(model, img_size=32, channels=1, n=16):
    """Generate images using original DDPM sampling (1000 steps)."""
    model.eval()
    x = torch.randn(n, channels, img_size, img_size).to(device)

    for i in tqdm(reversed(range(T)), total=T, desc="DDPM Sampling (1000 steps)"):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        x = p_sample_ddpm(model, x, t, i)

    x = (x.clamp(-1, 1) + 1) / 2
    return x

---
## Step 8 — Generate and Compare DDPM vs DDIM

In [ ]:
# ── Speed comparison ─────────────────────────────────────
print("Generating with DDPM (1000 steps)...")
t0 = time.time()
ddpm_samples = sample_ddpm(model, img_size=32, channels=1, n=16)
ddpm_time = time.time() - t0
print(f"DDPM time: {ddpm_time:.1f}s")

print("\nGenerating with DDIM (50 steps)...")
t0 = time.time()
ddim_samples_50 = sample_ddim(model, img_size=32, channels=1, n=16, ddim_steps=50)
ddim_time = time.time() - t0
print(f"DDIM time: {ddim_time:.1f}s")

print(f"\nSpeedup: {ddpm_time/ddim_time:.1f}x faster")

In [ ]:
# ── Visualize DDPM vs DDIM side by side ──────────────────
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

for i in range(8):
    # DDPM row
    axes[0, i].imshow(ddpm_samples[i, 0].cpu().numpy(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('DDPM\n(1000 steps)', fontsize=9)

    # DDIM row
    axes[1, i].imshow(ddim_samples_50[i, 0].cpu().numpy(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('DDIM\n(50 steps)', fontsize=9)

plt.suptitle(f'DDPM ({ddpm_time:.1f}s)  vs  DDIM ({ddim_time:.1f}s) — {ddpm_time/ddim_time:.1f}x speedup',
             fontsize=12)
plt.tight_layout()
plt.savefig('ddpm_vs_ddim.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ddpm_vs_ddim.png")

---
## Step 9 — Try Different DDIM Step Counts

DDIM can use as few as 10 steps — let's compare quality vs speed.

In [ ]:
# ── Compare different step counts ────────────────────────
step_counts = [10, 25, 50, 100]
results = {}

for steps in step_counts:
    t0 = time.time()
    samples = sample_ddim(model, img_size=32, channels=1, n=8, ddim_steps=steps)
    elapsed = time.time() - t0
    results[steps] = (samples, elapsed)
    print(f"Steps={steps:4d} | Time: {elapsed:.2f}s")

# Visualize
fig, axes = plt.subplots(len(step_counts), 8, figsize=(16, 2*len(step_counts)))

for row, steps in enumerate(step_counts):
    samples, elapsed = results[steps]
    for col in range(8):
        axes[row, col].imshow(samples[col, 0].cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_title(f'DDIM {steps} steps\n({elapsed:.1f}s)', fontsize=8)

plt.suptitle('DDIM Quality vs Speed Tradeoff', fontsize=13)
plt.tight_layout()
plt.savefig('ddim_steps_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ddim_steps_comparison.png")

---
## Summary

### What you built:
| Component | Status |
|---|---|
| Noise schedule | ✅ same as DDPM |
| U-Net model | ✅ same as DDPM |
| Training loop | ✅ same as DDPM |
| `ddim_step()` | ✅ NEW — estimate x0 + jump |
| `sample_ddim()` | ✅ NEW — skip timesteps |

### Key takeaway:
> DDIM changes **only 2 things** — the timestep list (skip steps) and the per-step formula (estimate x0 first, no random noise). Everything else is identical to DDPM.

### Next steps:
- Try **Classifier-Free Guidance (CFG)** for conditional generation
- Scale up to **CIFAR-10** (set `img_channels=3`)
- Read the **Latent Diffusion Model (LDM)** paper — how Stable Diffusion works